## Gold Daily Revenue

In [0]:
from pyspark.sql.functions import *
import pyspark.sql.functions as F

In [0]:
catalog ='ecommerce'
schema ='raw'
volume ='raw_data'

## gold_daily_revenue

In [0]:
%sql
create or replace table ecommerce.raw.gold_daily_revenue 
using delta 
as 
select o.order_date,
        p.category, 
        p.sub_category,
        o.payment_method,
        sum(o.qty) as total_qty,
        round(sum(o.qty * o.unit_price) , 2) as total_revenue
        
from ecommerce.raw.slv_orders o
join ecommerce.raw.slv_products p
on o.product_id = p.product_id
group by order_date, category, sub_category, payment_method
order by order_date, category, sub_category, payment_method;



num_affected_rows,num_inserted_rows


In [0]:
%sql
select count(*) from ecommerce.raw.gold_daily_revenue;

count(*)
9480


In [0]:
%sql
DESCRIBE TABLE ecommerce.raw.gold_daily_revenue;

col_name,data_type,comment
order_date,date,null
category,string,null
sub_category,string,null
payment_method,string,null
total_qty,bigint,null
total_revenue,double,null


## Gold Product Performance

In [0]:
%sql
create or replace table ecommerce.raw.gold_product_performance
using delta
as 
select 
        p.product_id,
        p.category,
        p.sub_category,
        p.rating,
        p.review_count,
        o.order_status,
        sum(o.qty) as total_quantity,
        round(sum(o.qty * o.unit_price),2) as total_revenue
from ecommerce.raw.slv_products p
join ecommerce.raw.slv_orders o
on p.product_id = o.product_id
group by p.product_id, p.category, p.sub_category, p.rating, p.review_count, o.order_status
order by total_revenue;

num_affected_rows,num_inserted_rows


In [0]:
%sql
Describe table ecommerce.raw.gold_product_performance;

col_name,data_type,comment
product_id,string,null
category,string,null
sub_category,string,null
rating,double,null
review_count,int,null
order_status,string,null
total_quantity,bigint,null
total_revenue,double,null


In [0]:
%sql
select count(*) from ecommerce.raw.gold_product_performance;

count(*)
3679


## Gold Customer 360

In [0]:
%sql

create or replace table ecommerce.raw.gold_customer_segmentation
using delta
as 
select c.customer_id, 
        c.customer_name,
        c.email,
        c.phone,
        c.state,
        c.pincode,
        c.signup_channel,
        c.customer_segment,
        c.is_active,
        count(distinct o.order_id) as total_orders,
        sum(o.qty) as total_qty,
        round(sum(o.qty * o.unit_price),2) as total_spend,
        round(sum(o.qty * o.unit_price)/ count(distinct o.order_id),2) as avg_order_value,
        min(o.order_date) as first_order_date,
        max(o.order_date) as last_order_date,
        sum(case when o.order_status = 'Delivered' then 1 else 0 end) as delivered_orders,
        sum(case when o.order_status = 'Cancelled' then 1 else 0 end) as cancelled_orders,
        sum(case when o.order_status = 'Shipped' then 1 else 0 end) as shipped_orders
from ecommerce.raw.slv_orders o
left join ecommerce.raw.slv_customers c
on o.customer_id = c.customer_id
group by c.customer_id, 
        c.customer_name,
        c.email,
        c.phone,
        c.state,
        c.pincode,
        c.signup_channel,
        c.customer_segment,
        c.is_active
order by total_spend desc



num_affected_rows,num_inserted_rows


In [0]:
%sql
Describe table ecommerce.raw.gold_customer_segmentation;

col_name,data_type,comment
customer_id,string,null
customer_name,string,null
email,string,null
phone,string,null
state,string,null
pincode,int,null
signup_channel,string,null
customer_segment,string,null
is_active,boolean,null
total_orders,bigint,null


In [0]:
%sql
select count(*) from ecommerce.raw.gold_customer_segmentation;

count(*)
4321


In [0]:
%sql


SELECT
    SUM(delivered_orders) AS total_delivered,
    SUM(cancelled_orders) AS total_cancelled,
    SUM(shipped_orders) AS total_shipped
FROM ecommerce.raw.gold_customer_segmentation;

total_delivered,total_cancelled,total_shipped
2524,2536,2518


In [0]:
%sql
SELECT COUNT(*) FROM ecommerce.raw.slv_orders;


COUNT(*)
10028


In [0]:
%sql
SELECT COUNT(*) FROM ecommerce.raw.slv_orders_autoloader;

COUNT(*)
6


In [0]:
%sql

SELECT COUNT(*) FROM ecommerce.raw.slv_order_streaming;

COUNT(*)
1445


In [0]:
%sql
SELECT * FROM ecommerce.raw.slv_orders LIMIT 5;

order_id,customer_id,product_id,order_date,qty,unit_price,payment_method,order_status
ORD28184,CUST00004,PROD00005,2026-09-05,5,5500.0,null,Shipped
ORD85690,CUST00001,PROD00003,2026-09-05,4,3500.0,null,Delivered
ORD10817,CUST00001,PROD00002,2026-09-05,2,2500.0,null,Cancelled
ORD21780,CUST00004,PROD00001,2026-09-05,5,1500.0,null,Shipped
ORD73258,CUST00002,PROD00004,2026-09-05,4,4500.0,null,Cancelled


In [0]:
%sql
SELECT * FROM ecommerce.raw.slv_orders_autoloader LIMIT 5;

order_id,customer_id,product_id,order_date,qty,unit_price,payment_method,order_status,_rescued_data
ORD90003,CST0386,PRD0509,2026-09-04,3,449794.29,Credit Card,Processing,null
ORD90002,CST0093,PRD0289,2026-09-04,1,484419.64,Net Banking,Delivered,null
ORD90004,CST2782,PRD0237,2026-09-04,4,45392.84,Debit Card,Shipped,null
ORD90001,CST1479,PRD0315,2026-09-04,2,67666.1,UPI,Shipped,null
ORD90005,CST3041,PRD0156,2026-09-04,2,259678.66,UPI,Delivered,null


In [0]:
%sql
SELECT * FROM ecommerce.raw.slv_order_streaming LIMIT 5;

event_id,event_ts,order_id,customer_id,product_id,order_date,qty,unit_price,payment_method,order_status
ed3823df-62dc-4e3f-9056-5fca6ba3df59,2026-09-05T10:41:33.704Z,ORD54110,CUST00005,PROD00002,2026-09-05,4,2500.0,null,Delivered
ad3248d9-4f65-41b9-9e96-7a077b246abc,2026-09-05T10:41:39.235Z,ORD48023,CUST00003,PROD00001,2026-09-05,1,1500.0,null,Shipped
d173c5bb-b1c5-4977-80c4-e16b16339b81,2026-09-05T10:41:41.498Z,ORD28184,CUST00004,PROD00005,2026-09-05,5,5500.0,null,Shipped
ecef6dba-b4b4-4d71-8386-1897f084326c,2026-09-05T10:41:43.768Z,ORD31370,CUST00005,PROD00001,2026-09-05,1,1500.0,null,Delivered
c7743d3a-938f-469c-9f44-1d1a5abc1984,2026-09-05T10:41:46.032Z,ORD56776,CUST00002,PROD00001,2026-09-05,5,1500.0,null,Shipped


#### Create gold table

In [0]:
%sql
CREATE TABLE ecommerce.raw.gold_orders (
    order_id STRING,
    customer_id STRING,
    product_id STRING,
    order_date DATE,
    qty INT,
    unit_price DOUBLE,
    payment_method STRING,
    order_status STRING,
    order_revenue DOUBLE,
    updated_at TIMESTAMP
);

In [0]:
%sql
SELECT COUNT(*)
FROM ecommerce.raw.gold_orders;

COUNT(*)
0


#### Loading existing silver data into table

In [0]:
%sql
INSERT INTO ecommerce.raw.gold_orders
SELECT
    order_id,
    customer_id,
    product_id,
    order_date,
    qty,
    unit_price,
    payment_method,
    order_status,
    qty * unit_price AS order_revenue,
    current_timestamp() AS updated_at
FROM ecommerce.raw.slv_orders;

num_affected_rows,num_inserted_rows
10028,10028


In [0]:
%sql
SELECT COUNT(*)
FROM ecommerce.raw.gold_orders;

COUNT(*)
10028


In [0]:
def process_batch(batch_df, batch_id):

    # batch_df = records arriving from Kafka

    # 1. Update Silver
    silver = DeltaTable.forName(
        spark,
        "ecommerce.raw.slv_orders"
    )

    (
        silver.alias("target")
        .merge(
            batch_df.alias("source"),
            "target.order_id = source.order_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    # 2. Prepare Gold
    gold_batch = (
        batch_df
        .select(
            "order_id",
            "customer_id",
            "product_id",
            "order_date",
            "qty",
            "unit_price",
            "payment_method",
            "order_status"
        )
        .withColumn(
            "order_revenue",
            col("qty") * col("unit_price")
        )
        .withColumn(
            "updated_at",
            current_timestamp()
        )
    )

    # 3. Update Gold
    gold = DeltaTable.forName(
        spark,
        "ecommerce.raw.gold_orders"
    )

    (
        gold.alias("target")
        .merge(
            gold_batch.alias("source"),
            "target.order_id = source.order_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
%sql
DESCRIBE TABLE ecommerce.raw.gold_orders;

col_name,data_type,comment
order_id,string,null
customer_id,string,null
product_id,string,null
order_date,date,null
qty,int,null
unit_price,double,null
payment_method,string,null
order_status,string,null
order_revenue,double,null
updated_at,timestamp,null


In [0]:
%sql
SELECT COUNT(*) 
FROM ecommerce.raw.gold_orders;

COUNT(*)
10028
